<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# Orbital Debris SQL Queries

This notebook is intentionally SQL-first and contains query work only (no visualizations).

## **Setup And Function Declarations**

In [ ]:
import pandas as pd
import sqlite3 as sql
import utility as utils

pd.set_option('display.max_columns', None)
orbital_debris_conn = sql.connect('../data/clean/orbital_debris.db')

# Sanity Check - Verify that the database connection is working and that we can retrieve data
sanity_check = utils.query_all_satellites(orbital_debris_conn)

sanity_check

## Primary Question 1: Growth and Decoupling Baseline
At what year does the present day in-orbit population stop following a legacy linear pattern and transition into a modern accelerating pattern?

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
# CTE yearly aggregates the number of objects in orbit, launch missions, and payloads in orbit by launch year,
# then calculates the share of the payloads and the cumulative number of objects in orbit over time.
q1 = '''
WITH yearly AS (
  SELECT
    launch_events.launch_year AS launch_year,
    COUNT(*) AS objects_in_orbit,
    COUNT(DISTINCT launch_events.launch_id) AS launch_missions,
    SUM(
      CASE
        WHEN UPPER(COALESCE(satellites.object_type, '')) = 'PAYLOAD' THEN 1
        ELSE 0
      END
    ) AS payload_in_orbit
  FROM satellites
  JOIN launch_events ON launch_events.launch_id = satellites.launch_id
  WHERE launch_events.launch_year IS NOT NULL
    AND COALESCE(satellites.in_orbit, 0) = 1
  GROUP BY launch_events.launch_year
)
SELECT
  launch_year,
  objects_in_orbit,
  launch_missions,
  payload_in_orbit,
  ROUND(100.0 * payload_in_orbit / NULLIF(objects_in_orbit, 0), 2) AS payload_share_pct,
  SUM(objects_in_orbit) OVER (ORDER BY launch_year) AS cumulative_in_orbit
FROM yearly
ORDER BY launch_year;
'''
pq1 = utils.run_query(q1, orbital_debris_conn)
pq1.to_parquet('../data/clean/results/orbit_trends_pq1.parquet', index=False)
pq1.head(10)

## **Primary Question 2: High-Risk Distribution by Altitude Band**
How are high-risk objects (by velocity and kinetic energy) distributed across orbit classes and altitude bands, especially in the 400 - 600 km LEO?

In [ ]:
# This query is a bit more complex, but it is designed to identify the highest risk objects in orbit 
# based on their kinetic energy.
#
# CTE banded: joins the orbital_data, risk_assessment, and satellites tables to
# get all of the relevent information/fields in one place and we create a new field called altitude_band
# that categorizes each object into one of the 5 altitude bands based on their perigee_km.
#
# CTE ranked: uses the NTILE window function to divide the objects into 4 quartiles based on their kinetic energy.
#   QUARTILE 1 = highest kinetic energy (most dangerous, top 25% of objects),
#   QUARTILE 2 = next highest kinetic energy (second most dangerous, 25-50% of objects),
#   QUARTILE 3 = moderate kinetic energy (50-75% of objects), 
#   QUARTILE 4 = lowest kinetic energy (least dangerous, bottom 25% of objects)
# Finally we select from ranked and group by orbit_class and altitude_band 
# to get the total number of objects, the number of high risk objects (those in the top quartile), the percentage 
# of high risk objects, the average velocity and kinetic energy of the high risk objects, the total kinetic energy
# of the high risk objects, and the number of high risk objects that are zombies, debris, payloads, or rocket bodies.

q2 = '''
WITH banded AS (
  SELECT
    orbital_data.norad_id,
    orbital_data.orbit_class,
    orbital_data.perigee_km,
    orbital_data.proxy_mass_kg,
    risk_assessment.velocity_kms,
    risk_assessment.kinetic_joules,
    risk_assessment.is_zombie,
    satellites.object_type,
    CASE
      WHEN orbital_data.perigee_km < 400 THEN '1. < 400 km'
      WHEN orbital_data.perigee_km >= 400  AND orbital_data.perigee_km < 600 THEN '2. 400-600 km (Kessler Zone)'
      WHEN orbital_data.perigee_km >= 600  AND orbital_data.perigee_km < 1000 THEN '3. 600-1000 km'
      WHEN orbital_data.perigee_km >= 1000 AND orbital_data.perigee_km < 2000 THEN '4. 1000-2000 km'
      ELSE '5. > 2000 km'
    END AS altitude_band
  FROM orbital_data
  JOIN risk_assessment ON risk_assessment.norad_id = orbital_data.norad_id
  JOIN satellites ON satellites.norad_id = orbital_data.norad_id
  WHERE risk_assessment.kinetic_joules IS NOT NULL
    AND orbital_data.perigee_km IS NOT NULL
),
ranked AS (
  SELECT
    *,
    NTILE(4) OVER (ORDER BY kinetic_joules DESC) AS kinetic_quartile
  FROM banded
)
SELECT
  orbit_class,
  altitude_band,

  COUNT(*) AS total_objects,
  SUM(CASE WHEN kinetic_quartile = 1 THEN 1 ELSE 0 END) AS high_risk_count,
  
  ROUND(100.0 * SUM(CASE WHEN kinetic_quartile = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS high_risk_pct,
  ROUND(AVG(CASE WHEN kinetic_quartile = 1 THEN velocity_kms  END), 4) AS avg_velocity_kms,
  ROUND(AVG(CASE WHEN kinetic_quartile = 1 THEN kinetic_joules END), 0) AS avg_kinetic_joules,
  ROUND(SUM(CASE WHEN kinetic_quartile = 1 THEN kinetic_joules ELSE 0 END), 0) AS total_kinetic_joules,
  
  SUM(CASE WHEN kinetic_quartile = 1 AND is_zombie = 1 THEN 1 ELSE 0 END) AS zombie_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'DEBRIS'  THEN 1 ELSE 0 END) AS debris_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'PAYLOAD' THEN 1 ELSE 0 END) AS payload_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'ROCKET BODY' THEN 1 ELSE 0 END) AS rb_high_risk
FROM ranked
GROUP BY orbit_class, altitude_band
ORDER BY altitude_band, orbit_class;
'''

high_risk = utils.run_query(q2, orbital_debris_conn)
high_risk.to_parquet('../data/clean/results/high_risk_pq2.parquet', index=False)
high_risk.head(10)

## **Primary Question 3: Zombie Concentration by Owner**

## **Secondary: Object Type × Operational Status**

## **Secondary: User Category Profile**

## **Extra Questions!**

In [ ]:
eq1 = '''
SELECT
    satellites.norad_id,
    satellites.object_name,
    launch_events.launch_year,
    satellites.object_type,
    orbital_data.orbit_class,
    CASE
      WHEN orbital_data.perigee_km < 400 THEN '< 400 km'
      WHEN orbital_data.perigee_km >= 400  AND orbital_data.perigee_km < 600 THEN '400-600 km (Kessler Zone)'
      WHEN orbital_data.perigee_km >= 600  AND orbital_data.perigee_km < 1000 THEN '600-1000 km'
      WHEN orbital_data.perigee_km >= 1000 AND orbital_data.perigee_km < 2000 THEN '1000-2000 km'
      ELSE '> 2000 km'
    END AS altitude_band,
    CASE
        WHEN risk_assessment.kinetic_joules >= 1e12 THEN 'Extremely High Risk'
        WHEN risk_assessment.kinetic_joules >= 1e10 THEN 'High Risk'
        WHEN risk_assessment.kinetic_joules >= 1e8 THEN 'Moderate Risk'
        ELSE 'Low Risk'
    END AS risk_level,
    CASE
        WHEN ownership_operators.is_commercial = 1 THEN 'Commercial'
        WHEN ownership_operators.is_government = 1 THEN 'Government'
        WHEN ownership_operators.is_military = 1 THEN 'Military'
        WHEN ownership_operators.is_civil = 1 THEN 'Civilian'
        ELSE 'Unknown'
    END AS sector
FROM satellites
JOIN orbital_data ON orbital_data.norad_id = satellites.norad_id
JOIN risk_assessment ON risk_assessment.norad_id = satellites.norad_id
JOIN ownership_operators ON satellites.owner_code = ownership_operators.owner_code
JOIN launch_events ON launch_events.launch_id = satellites.launch_id
WHERE risk_assessment.kinetic_joules IS NOT NULL;
'''

hr_top = utils.run_query(eq1, orbital_debris_conn)
hr_top.to_parquet('../data/clean/results/alt_sector_risk_bands_eq1.parquet')
hr_top